# Einstein metrics on CP⁴ — a cell-by-cell demonstration

Run from top to bottom with **Shift+Enter**, using the **SageMath** kernel.
Each original definition cell is followed by a short **Example** that calls its functions
and displays a result. The definition cells now print a brief readiness message too.
Wait until the execution indicator stops showing `[*]` before moving on.

The examples use the paper's **n = 4** parameters. Coefficient formulas are exact;
numerical ball outputs enclose their true values. A number followed by `?` is Sage's
ball notation: its last displayed digit is uncertain. These intermediate results
do not by themselves certify a zero; the final example performs the full check.

Examples 6–8 perform rigorous ODE integrations and can take longer. The earlier full
certification took about 101 seconds on Binder, excluding setup; timings vary.
All outputs start empty and are produced by your own execution. The scientific
functions and the final certification parameters are unchanged.

In [ ]:
#=======================================================
#=======
# CELL 1
#=======
# In this cell, we compute SYMBOLICALLY the first coefficients given by the formal series
# corresponding to Subsection 5.1 (Lemma 5.2)
# The series is computed in terms of b2, u1 and (for each n) every term will be given
# by rational polynomials in (b2, u1) 
#========================================================  

# We will save the series coefficients globally. In order to avoid recomputing
# we store variables n_stored_rp (dimension) and K_ORD_stored_rp (order of the series)
# and only recompute when we need to change those parameters.
# We initialized n_stored_rp = -1 and K_ORD_stored_rp = -1 to indicate no series has been computed

n_stored_rp = -1
K_ORD_stored_rp = -1
def rp_coefficients(n, K_ORD):
    # We will store the coefficients globally and rp_coefficients just computes them
    global a_rp, b_rp, u_rp, v_rp, N1_rp, N2_rp, D_rp, Q_rp, b2, u1, n_stored_rp, K_ORD_stored_rp

    # If the coefficients with this n are already computed, we skip
    if n_stored_rp == n and K_ORD_stored_rp >= K_ORD:
        return

    # Each coefficient is a polynomial of rational coefficients on (b2, u1) [n has been fixed]
    R = PolynomialRing(QQ, names=('b2', 'u1'))
    b2, u1 = R.gens()

    # We impose K_ORD odd. The series will be stored as arrays a_rp, b_rp, u_rp, v_rp, N1_rp, N2_rp, D_rp and Q_rp
    # indexed by their corresponding order 
    if (K_ORD%2 == 0) or (K_ORD < 3):
        print("XXXXXXXXXXX \n ERROR: K_ORD order invalid \n XXXXXXXXXXXXX")

    a_rp = {}; b_rp = {}; u_rp = {}; v_rp = {}
    N1_rp = {}; N2_rp = {}; D_rp = {}; Q_rp = {}

    # We initialize the first coefficients at RP^n according to Lemma 5.2 (57).
    # From the smoothness conditions (49)
    a_rp[0] = 1
    v_rp[-1] = -1/(n-1)
    
    # We take b2 and u1 as our parameters describing the solution, as in Lemma 5.2
    b_rp[2] = b2
    u_rp[1] = u1
    
    # The values of a_rp[2] and v_rp[1] are given in (57)
    a_rp[2] = -(n-1)*u1
    v_rp[1] = (2*n-1) / (n*(n-1)) * b2 + (n-1)/n * u1 - 2*(n+1)*(2*n-1)/n
    
    # We take the initial values of Q, D, N1, N2 from (56)
    Q_rp[-2] = 1/(n-1)
    Q_rp[0] = 2*(n+1) - (2*n*v_rp[1]-2*(n-1)*u1) / (2*n-1)
    D_rp[0] = 2*(n-2)
    N1_rp[0] = 0
    N2_rp[0] = 2*(n-2)
    
    # We use (55) to compute D_rp[2], N1_rp[2] and N2_rp[2]
    D_rp[2]  = 2*(n-2)*a_rp[2] + 2*n*b_rp[2]
    N1_rp[2] = 2*n*a_rp[2]
    N2_rp[2] = (4*n-6)*a_rp[2] - 2*(n-1)*b_rp[2]
    
    # ------------------------------------------------
    # We compute iteratively the coefficients at RP^n up to order K_ORD using the recursions
    # of Lemma 5.2: first  b_{k+1}, then (a_{k+1}, u_k), and lastly v_k (60).
    # Also involves computing D, N1, N2, Q and N1^des at every order
    
    for k in range(3, K_ORD+1, 2):
        # We compute b_rp[k+1] using (58)
        Rbk = -2*sum(b_rp[i]*v_rp[k-i] for i in range(2, k, 2)) \
              - sum(j*Q_rp[k-1-j]*b_rp[j] for j in range(2, k, 2))
        b_rp[k+1] = (n-1)/(k-1)*Rbk
    
        # We compute N1des[k+1] (since we only need it at level k+1 we do not store it in an array)
        N1des = n*sum(a_rp[i]*a_rp[k+1-i] for i in range(2, k, 2)) \
                + (n-2)*sum(b_rp[i]*b_rp[k+1-i] for i in range(2, k, 2)) \
                - 2*(n-1)*sum(a_rp[i]*b_rp[k+1-i] for i in range(0, k, 2)) \
                + 2*(n-1)*b_rp[k+1]
        
        # We compute a_rp[k+1} and u_rp[k] using (59)
        R_a = 2*sum(a_rp[i]*u_rp[k-i] for i in range(2, k, 2)) \
              + sum(j*Q_rp[k-1-j]*a_rp[j] for j in range(2, k, 2))
    
        R_u = (n-1)*( (n-2)/(n-1)**2*N1des + 2*(2*n-1)*(n+1)/(n-1)*N1_rp[k-1]
                      - 1/(n-1)*sum(Q_rp[i]*N1_rp[k-1-i] for i in range(0, k-1, 2))
                      + Q_rp[-2]*sum(j*D_rp[k-j]*u_rp[j] for j in range(1, k-1, 2))
                      + sum(j*D_rp[i]*Q_rp[l]*u_rp[j]
                            for i in range(2, k, 2) for l in range(0, k-1, 2)
                            for j in [k-2-i-l] if 1 <= j <= k-2)
                      + D_rp[0]*sum(j*Q_rp[k-2-j]*u_rp[j] for j in range(1, k-1, 2))
                      + sum(D_rp[i]*u_rp[k-i] for i in range(2, k, 2)) )
    
        a_rp[k+1] = (-(n-1)*(k+n-1)*R_a + (n-1)/(n-2)*R_u)/((k-1)*(k+n+1))
        u_rp[k]   = (n*R_a - (k+1)/(2*(n-2))*R_u)/((k-1)*(k+n+1))
    
        # Now, we proceed to compute the new N1, N2, D from (55)
        N1_rp[k+1] = N1des + 2*n*a_rp[k+1]
        D_rp[k+1] = 2*(n-1)*(a_rp[k+1] + b_rp[k+1]) \
                    + 2*sum(a_rp[i]*b_rp[k+1-i] for i in range(0, k, 2)) \
                    - sum(a_rp[i]*a_rp[k+1-i] for i in range(0, k+2, 2)) \
                    - sum(b_rp[i]*b_rp[k+1-i] for i in range(2, k, 2))
        N2_rp[k+1] = (n-2)*sum(a_rp[i]*a_rp[k+1-i] for i in range(0, k+2, 2)) \
                     + n*sum(b_rp[i]*b_rp[k+1-i] for i in range(2, k, 2)) \
                     - 2*(n-1)*sum(a_rp[i]*b_rp[k+1-i] for i in range(0, k, 2)) \
                     + 2*(n-1)*a_rp[k+1]
    
        # We compute v_rp[k] using (60)
        Rvk = -(n-1)*(D_rp[k+1]*v_rp[-1] + sum(D_rp[i]*v_rp[k-i] for i in range(2, k, 2))) \
            - 2*(2*n-1)*(n+1)*N2_rp[k-1] - N2_rp[k+1] \
            + Q_rp[-2]*N2_rp[k+1] + sum(Q_rp[i]*N2_rp[k-1-i] for i in range(0, k-1, 2)) \
            + (n-1)*( Q_rp[-2]*D_rp[k+1]*v_rp[-1] \
            + v_rp[-1]*sum(D_rp[i]*Q_rp[k-1-i] for i in range(2, k, 2)) \
            - Q_rp[-2]*sum(l*D_rp[k-l]*v_rp[l] for l in range(1, k-1, 2)) \
            - D_rp[0]*sum(l*Q_rp[k-2-l]*v_rp[l] for l in range(1, k-1, 2)) \
            - sum(l*D_rp[i]*Q_rp[jj]*v_rp[l]
                              for i in range(2, k+2, 2) for jj in range(0, k-1, 2)
                              for l in [k-2-i-jj] if 1 <= l <= k-2) )
        v_rp[k] = Rvk/(2*(n-2)*(k+n-1))
    
        # And finally, we update Q_rp[k-1] using (55)
        Q_rp[k-1] = (n*(n-1)*(sum(u_rp[i]*u_rp[k-1-i] for i in range(1, k-1, 2))
                              + sum(v_rp[i]*v_rp[k-1-i] for i in range(-1, k+1, 2)))
                     - 2*(n-1)**2*sum(u_rp[i]*v_rp[k-1-i] for i in range(1, k+1, 2)))/(2*n-1)

    # We update n_stored_rp
    print("RP coefficients computed for n = ", n, " up to order K_ORD = ", K_ORD)
    n_stored_rp = n
    K_ORD_stored_rp = K_ORD

print("CELL 1: definitions ready. Run Example 1 below.", flush=True)

## Example 1 — see the formal coefficients
Compute the real-projective-end expansion for **n = 4** to the order used in the
certification. Display a few low-order coefficients as exact polynomials in
the two free parameters. In particular, $a_2=-3u_1$.

In [ ]:
demo_n = 4
demo_m_rp = 35
demo_m_qc = 16
print("Computing the RP coefficients for n = 4 ...", flush=True)
rp_coefficients(demo_n, demo_m_rp)
for demo_label, demo_value in (("a_2", a_rp[2]), ("b_2", b_rp[2]),
                              ("u_1", u_rp[1]), ("v_1", v_rp[1]),
                              ("a_4", a_rp[4])):
    print(demo_label + " =")
    show(demo_value)

In [ ]:
# CELL 2
# This cell is analogous to CELL 1 but following Subsection 5.3 and Lemma 5.6, doing the expansion
# at the complex quadric in terms of (alm2, bem1) = (alpha_{-2}, beta_{-1}). As a minor difference,
# we also express things in terms of w, which stands for 1/D_{-2} with D_{-2} = 4((n-1)*alm2 - bem1^2)
# This keeps our expressions simpler and polynomial (except for negative powers of alm2). w will later be 
# substituted by its corresponding value.
#========================================================

n_stored_qc = -1
K_ORD_stored_qc = -1
def qc_coefficients(n, K_ORD, base=QQ):
    
    global al_qc, be_qc, mu_qc, eta_qc, D_qc, Q_qc, Np_qc, Nm_qc, alm2, bem1, w, n_stored_qc, K_ORD_stored_qc

    if n_stored_qc == n and K_ORD <= K_ORD_stored_qc:
        return

    L = LaurentPolynomialRing(base, names=('alm2', 'bem1', 'w'))
    alm2, bem1, w = L.gens()
    al_qc = {}; be_qc = {}; mu_qc = {}; eta_qc = {}
    D_qc = {}; Q_qc = {}; Np_qc = {}; Nm_qc = {}
    # We initialize the first coefficients at the quadric according to Lemma 5.6 (75).
    al_qc[-2] = alm2
    be_qc[-1] = bem1
    mu_qc[-1] = 1
    eta_qc[0]  = -bem1*alm2**(-1)/2
    mu_qc[1]  = (2*n-1)*(n+1) - (2*n-1)*(n-1)*alm2**(-1)/8

    # We take the initial values of Q, D, N+, N- from (74)
    Q_qc[-2]  = 1
    Q_qc[0]   = 2*(n+1) + 4*(n-1)/(2*n-1)*mu_qc[1] + 2*(n-1)*eta_qc[0]**2
    D_qc[-2]  = 4*(n-1)*alm2 - 4*bem1**2
    Np_qc[-2] = 2*(n-1)*alm2 + 4*(n-1)*bem1**2
    Nm_qc[-3] = 4*alm2*bem1

    # ------------------------------------------------
    # We compute iteratively the coefficients at the quadric up to order K_ORD using the
    # recursions of Lemma 5.6: first alpha_m, then (eta_{m+2}, mu_{m+3}) from the 2x2
    # system after eliminating beta_{m+1}, and lastly beta_{m+1}, as well as their 'des' analogues

    for m in range(0, K_ORD+1, 2):
        # We compute al_qc[m] using (76):  alpha_m = -R_alpha/(m+2)
        R_alpha = sum(j*Q_qc[m-2-j]*al_qc[j] for j in range(-2, m-1, 2)) \
                  + 2*sum(al_qc[i]*mu_qc[m-1-i] for i in range(-2, m-1, 2)) \
                  + 2*sum(be_qc[i]*eta_qc[m-1-i] for i in range(-1, m, 2))
        al_qc[m] = -R_alpha/(m+2)

        # We compute the D_m^des, N_+m^des, N_-m^des and Q^des_m from (83) 
        Ddes = 4*(n-1)*al_qc[m] - 4*sum(be_qc[i]*be_qc[m-i] for i in range(1, m, 2)) \
               - (1 if m == 0 else 0)
        Npdes = 2*(n-1)*al_qc[m] + 4*(n-1)*sum(be_qc[i]*be_qc[m-i] for i in range(1, m, 2)) \
                - (n if m == 0 else 0)
        Nmdes = 4*sum(al_qc[i]*be_qc[m-1-i] for i in range(0, m+1, 2)) - 2*(n-1)*be_qc[m-1]
        Qdes = (2*(n-1)*sum(mu_qc[i]*mu_qc[m+2-i] for i in range(1, m+2, 2))
                + 2*(n-1)*(2*n-1)*sum(eta_qc[i]*eta_qc[m+2-i] for i in range(2, m+1, 2)))/(2*n-1)

        # We compute R_beta using (77)
        R_beta = sum(p*Q_qc[m-1-p]*be_qc[p] for p in range(-1, m, 2)) \
                 + 2*sum(al_qc[i]*eta_qc[m-i] for i in range(0, m+1, 2)) \
                 + 2*sum(be_qc[i]*mu_qc[m-i] for i in range(-1, m, 2))

        # We compute R_eta using (81)
        R_eta = (n-1)*sum(p*D_qc[ia]*Q_qc[ib]*eta_qc[p]
                            for ia in range(-2, m-1, 2) for ib in range(-2, m+1, 2)
                            for p in [m-2-ia-ib] if 0 <= p <= m) \
                  + (n-1)*(Ddes*eta_qc[0] + sum(D_qc[i]*eta_qc[m-i] for i in range(0, m-1, 2))) \
                  + (2*n-1)*2*(n+1)*Nmdes \
                  - (Q_qc[0]*Nmdes + Qdes*Nm_qc[-3]
                     + sum(Q_qc[i]*Nm_qc[m-1-i] for i in range(2, m+1, 2)))

        # We compute R_mu using (82); the three explicit corrections are the
        # (ia, ib) = (m, -2), (m, 0), (-2, m+2) terms of the double sum, which involve Ddes, Qdes
        R_mu = (n-1)*( sum(p*D_qc[ia]*Q_qc[ib]*mu_qc[p]
                            for ia in range(-2, m-1, 2) for ib in range(-2, m+1, 2)
                            for p in [m-1-ia-ib] if -1 <= p <= m+1)
                        + Ddes*(Q_qc[-2]*mu_qc[1] - Q_qc[0]*mu_qc[-1])
                        - D_qc[-2]*Qdes*mu_qc[-1] ) \
                + (n-1)*(sum(D_qc[i]*mu_qc[m+1-i] for i in range(0, m-1, 2)) + Ddes*mu_qc[1]) \
                + (2*n-1)*2*(n+1)*Npdes \
                - (Q_qc[0]*Npdes + Qdes*Np_qc[-2]
                   + sum(Q_qc[i]*Np_qc[m-i] for i in range(2, m+1, 2)))

        # c-coefficients from (80)
        c_etaeta = -4*(n-1)*((m+3)*(n-1)*alm2 - (m+1)*bem1**2)
        c_etamu = 16*(n-1)*alm2*bem1/(2*n-1)
        c_etabe = -2*(n-1)*((n-1)*alm2 + bem1**2)*alm2**(-1)
        c_mube = -2*(n-1)**2*(2*n-1)*bem1*alm2**(-1)
        c_mumu = -4*(n-1)*((m+2)*(n-1)*alm2 - (m+4)*bem1**2)
        c_mueta = -4*(n-1)**2*(2*n-1)*bem1

        # We eliminate beta_{m+1} via (78) and solve the 2x2 system (86),
        # whose determinant equals 16(m+2)^2(m+4)(n-1)^2((n-1)alm2 - bem1^2)^2/(m+3) due to (85)
        # so that dividing by it is multiplication by w^2 (m+3)/((m+2)^2 (m+4)(n-1)^2),  with w = 1/D_{-2}.
        eta_qc[m+2] = ( c_mumu*(R_eta + c_etabe*R_beta/(m+3))
                       - c_etamu*(R_mu + c_mube*R_beta/(m+3)) ) \
                     * w**2*(m+3)/((m+2)**2*(m+4)*(n-1)**2)
        mu_qc[m+3] = ( -(c_mueta - 2*alm2*c_mube/(m+3))*(R_eta + c_etabe*R_beta/(m+3))
                       + (c_etaeta - 2*alm2*c_etabe/(m+3))*(R_mu + c_mube*R_beta/(m+3)) ) \
                     * w**2*(m+3)/((m+2)**2*(m+4)*(n-1)**2)
        be_qc[m+1] = -(2*alm2*eta_qc[m+2] + R_beta)/(m+3)

        # We store the FULL auxiliaries at their top orders (later iterations consume them):
        D_qc[m]    = Ddes - 8*bem1*be_qc[m+1]
        Np_qc[m]   = Npdes + 8*(n-1)*bem1*be_qc[m+1]
        Nm_qc[m-1] = Nmdes + 4*alm2*be_qc[m+1]
        Q_qc[m+2]  = Qdes + 4*(n-1)/(2*n-1)*mu_qc[m+3] + 4*(n-1)*eta_qc[0]*eta_qc[m+2]

    print("QC coefficients succesfully computed at n = ", n, " and K_ORD = ", K_ORD)
    n_stored_qc = n
    K_ORD_stored_qc = K_ORD


print("CELL 2: definitions ready. Run Example 2 below.", flush=True)

## Example 2 — coefficients at the quadric end
Here the two free parameters are `alm2` ($\alpha_{-2}$) and `bem1`
($\beta_{-1}$). The first dependent coefficients are rational functions of them.

In [ ]:
print("Computing the quadric coefficients for n = 4 ...", flush=True)
qc_coefficients(demo_n, demo_m_qc)
for demo_label, demo_value in (("eta_0", eta_qc[0]), ("mu_1", mu_qc[1])):
    print(demo_label + " =")
    show(demo_value)

In [ ]:
# CELL 3
# We implement a function that returns True/False according to the hypothesis of the inductive Lemma 5.4
# If all hypothesis are satisfied for any (b2, u1, C_a, C_b, C_u, C_v, C_rad) in their corresponding 
# complex (for b2, u1) or real balls (for C_x), then our function returns True. Otherwise returns False
#==========

# We define real and complex ball fields for our interval arithmetic operations
RBF = RealBallField(256)
CBF = ComplexBallField(256)

def check_rp_constants(n, m, b2c, u1c, C_a, C_b, C_u, C_v, C_rad, verbose = True):

    # Part (i): standard checks
    if m%2==0 or m<3:
        if verbose: print("Incorrect value of m")
        return False

    if b2c.real().lower()<=0:
        if verbose: print("Non-positive b2 value")
        return False

    if C_rad.lower()<=1:
        if verbose: print("C_rad must be > 1")
        return False

    if C_a.lower() <= 0 or C_b.lower() <= 0 or C_u.lower() <= 0 or C_v.lower() <= 0 :
        if verbose: print("All C_x must be > 0")
        return False

    # Part (ii): We check the bounds of the first coefficients
    rp_coefficients(n, m-2)
    for k in range(1, m-1, 2):
        for nm_, val, Cx in (("a", a_rp[k+1], C_a), ("b", b_rp[k+1], C_b),
                             ("u", u_rp[k], C_u), ("v", v_rp[k], C_v)):
            s = abs(val(b2c, u1c))
            if not s.upper() <= (Cx*catalan_number(k)*C_rad**k).lower():
                if verbose: print("  FAIL base case: |%s| at k=%d vs C_%s Cat(k) C_rad^k"
                                  % (nm_, k, nm_))
                return False
            
    # Part (iii): We check the closure inequalities (65), (69) and (70)
    # We start computing CD, CNi, CQ, CN1des, CRu according to (63) and (69)
    CD  = 2*n*(C_a + C_b) + (C_a + C_b)**2/C_rad
    CN1 = 2*n*C_a + (n*C_a**2 + (n-2)*C_b**2 + 2*(n-1)*C_a*C_b)/C_rad
    CN2 = (4*n-6)*C_a + 2*(n-1)*C_b + ((n-2)*C_a**2 + n*C_b**2 + 2*(n-1)*C_a*C_b)/C_rad
    CQ  = (2*(n-1)*C_u + 2*n*C_v)/(2*n-1) \
          + (n*(n-1)*(C_u**2 + C_v**2) + 2*(n-1)**2*C_u*C_v)/((2*n-1)*C_rad) \
          + 2*(n+1)/C_rad
    CN1des = n*C_a**2 + (n-2)*C_b**2 + 2*(n-1)*C_a*C_b
    CRu = C_u*((n+1)/2*CD + (n-1)*CQ*(CD/C_rad + 2*(n-2))) \
          + (n-2)*CN1des/(2*(n-1)) + CN1*((2*n-1)*(n+1)/C_rad + CQ/2)

    # And now we check inequalities (65), (69) and (70)
    closures = (
        ("(65)         ", (n-1)*(C_v + CQ),                          C_rad),
        ("(69), first  ", (C_u + CQ)*C_a + CRu/((n-2)*(n+4)),        C_a*C_rad/(n-1)),
        ("(69), second ", n/(n+3)*C_a*(C_u + CQ) + CRu/(2*(n-2)),    C_u*C_rad),
        ("(70)         ", (n*CD/(5*(n-1)) + 2*(2*n-1)*(n+1)*CN2/(5*C_rad**2)
                           + (n-1)*CD*CQ*C_v/C_rad**2 + (CQ*CN2 + CD*CQ)/(5*C_rad)
                           + (CD*C_v + 2*(n-1)*(n-2)*CQ*C_v)/C_rad)/(2*(n-2))
                           + CN2/(10*(n-1)),                          C_v),
    )
    for name, Lhs, Rhs in closures:
        if not Lhs.upper() <= Rhs.lower():
            if verbose: print("  FAIL closure %s:  LHS up to %s  vs  RHS from %s"
                              % (name, Lhs.upper(), Rhs.lower()))
            return False
        elif verbose:
            print("  %s holds with margin %.3f" % (name, float(Rhs.lower()/Lhs.upper())))
    return True


def derive_rp_constants(n, m, b2c, u1c, verbose=True):
    # Here, we search for constants C_x satisfying the hypothesis of Proposition 5.5
    # We do it as follows. We first iterate dyadically on C_rad. Then we set C_x to satisfy
    # the bounds (62) for the coefficients at orders k<m. Finally, we enlarge C_v if needed
    # (see proof of Proposition 5.5). Then, we check if all the inequalities are satisfied by
    # calling check_rp_constants. If we succeed, we are done. Otherwise, we double C_rad and try again

    # We compute coefficients and bound their absolute value for any (b2, u1) \in (b2c, u1c) \subset C^2
    rp_coefficients(n, m)
    sups = {}
    for k in range(1, m-1, 2):
        sups[('a', k)] = abs(a_rp[k+1](b2c, u1c))
        sups[('b', k)] = abs(b_rp[k+1](b2c, u1c))
        sups[('u', k)] = abs(u_rp[k](b2c, u1c))
        sups[('v', k)] = abs(v_rp[k](b2c, u1c))

    # We do our dyadic iteration in C_rad.
    margin = RBF(11)/RBF(10)
    for e in range(1, 200):
        C_rad = RBF(1.5)**e
        Cs = {}

        # We start setting C_a, C_b, C_u, C_v to have bounds (62) with k < m
        for x in ('a', 'b', 'u', 'v'):
            Cs[x] = margin * RBF(max((sups[(x, k)]/(RBF(catalan_number(k))*C_rad**k)).upper()
                                   for k in range(1, m-1, 2)))
            
        # If needed, we enlarge C_v as in the last paragraph of the proof of Proposition 5.5
        CD  = 2*n*(Cs['a'] + Cs['b']) + (Cs['a'] + Cs['b'])**2/C_rad
        CN2 = (4*n-6)*Cs['a'] + 2*(n-1)*Cs['b'] \
              + ((n-2)*Cs['a']**2 + n*Cs['b']**2 + 2*(n-1)*Cs['a']*Cs['b'])/C_rad
        enlarge = (n*CD + (n-2)*CN2)/(5*(n-1)*(n-2))
        if not Cs['v'].lower() >= enlarge.upper():
            Cs['v'] = margin*RBF(enlarge.upper())

        # We check if our candidate configuration C_a, C_b, C_u, C_v, C_rad works
        if check_rp_constants(n, m, b2c, u1c, Cs['a'], Cs['b'], Cs['u'],
                              Cs['v'], C_rad, verbose=False):
            if verbose:
                print("SUCCESS at C_rad = 2^%d:  C_a=%.4g  C_b=%.4g  C_u=%.4g  C_v=%.4g"
                      % (e, float(Cs['a']), float(Cs['b']), float(Cs['u']), float(Cs['v'])))
                check_rp_constants(n, m, b2c.real(), u1c.real(), Cs['a'], Cs['b'],
                                   Cs['u'], Cs['v'], C_rad, verbose=True)  # print margins
            return Cs['a'], Cs['b'], Cs['u'], Cs['v'], C_rad
            
    if verbose: print("FAILED: no C_rad up to 2^30 works")
    return None

print("CELL 3: definitions ready. Run Example 3 below.", flush=True)

## Example 3 — verify the Catalan bounds at the RP end
Use the paper's candidate centre, a real box radius $\delta=10^{-11}$ and the
complex enlargement $r=10^{-4}$ used for parameter derivatives. Search for
constants and then check all the inequalities **on these complex balls**.
The displayed $C_{\rm rad}$ gives the sufficient condition
$|H|>4C_{\rm rad}$. It is a bound, not the exact convergence radius.

In [ ]:
demo_delta = RBF("1e-11")
demo_r = RBF("1e-4")
demo_Hm, demo_Hp = RBF(-800), RBF(1200)
demo_p0 = [RBF("39.073179213574893"), RBF("6.0328404911983641"),
           RBF("0.0559943268417511"), RBF("-0.042848398677701")]
demo_box = [value.add_error(demo_delta) for value in demo_p0]
demo_complex_box = [CBF(value).add_error(demo_r) for value in demo_box]
demo_rp_constants = derive_rp_constants(demo_n, demo_m_rp,
    demo_complex_box[0], demo_complex_box[1], verbose=False)
assert demo_rp_constants is not None, "No RP constants certified."
for demo_label, demo_value in zip(("C_a", "C_b", "C_u", "C_v", "C_rad"), demo_rp_constants):
    print(demo_label, "=", demo_value)
demo_rp_bounds_ok = check_rp_constants(demo_n, demo_m_rp,
    demo_complex_box[0], demo_complex_box[1], *demo_rp_constants, verbose=True)
assert demo_rp_bounds_ok
print("RP bounds verified on the complex parameter balls:", demo_rp_bounds_ok)
print("4 C_rad =", 4*demo_rp_constants[-1], "; chosen |H_-| =", abs(demo_Hm))
assert abs(demo_Hm).lower() > (4*demo_rp_constants[-1]).upper()

In [ ]:
# CELL 4.
# This cell is analogous to CELL 3 but for the complex quadric endpoint, thus implementing the conditions
# from Proposition 5.8 in Subsection 5.4 of the paper

def check_qc_constants(n, m, alm2c, bem1c, C_al, C_be, C_eta, C_mu, C_rad, verbose=True):
    
    # Part (i): standard checks
    if m % 2 == 1 or m < 2:
        if verbose: print("Incorrect value of m")
        return False
        
    if C_rad.lower() <= 1:
        if verbose: print("C_rad must be > 1")
        return False
        
    if C_al.lower() <= 0 or C_be.lower() <= 0 or C_eta.lower() <= 0 or C_mu.lower() <= 0:
        if verbose: print("All C_x must be > 0")
        return False
        
    if alm2c.contains_zero():
        if verbose: print("alpha_{-2} ball contains 0")
        return False
        
    Dm2c = 4*((n-1)*alm2c - bem1c**2)
    if Dm2c.contains_zero():
        if verbose: print("D_{-2} ball contains 0")
        return False

    # We recall that our coefficients are expressed as a linear combination of positive and negative
    # powers of alpha_{-2}, and positive powers of w := 1/D_{-2} and beta_{-1}. We define w
    wc = 1/Dm2c

    # Part (ii): We check the bounds (87) for k < m
    qc_coefficients(n, m)
    for k in range(0, m-1, 2):
        for nm_, val, Cx in (("alpha", al_qc[k], C_al), ("beta", be_qc[k+1], C_be),
                             ("eta", eta_qc[k+2], C_eta), ("mu", mu_qc[k+3], C_mu)):
            s = abs(val(alm2c, bem1c, wc))
            if not s.upper() <= (Cx*catalan_number(k)*C_rad**k).lower():
                if verbose: print("  FAIL base case: |%s| at k=%d vs C_%s Cat(k) C_rad^k"
                                  % (nm_, k, nm_))
                return False
                
    # Part (iii): We check the closure inequalities (89) and (94)-(96)
    lam = 2*(n+1)
    aA   = abs(alm2c); aB = abs(bem1c); aD2 = abs(Dm2c)
    aMu1  = abs(mu_qc[1](alm2c, bem1c, wc));  aEta0 = abs(eta_qc[0](alm2c, bem1c, wc))
    aQ0  = abs(Q_qc[0](alm2c, bem1c, wc));   aD0 = abs(D_qc[0](alm2c, bem1c, wc))
    aNp0 = abs(Np_qc[0](alm2c, bem1c, wc));  aNm1 = abs(Nm_qc[-1](alm2c, bem1c, wc))
    aNpm2 = abs(Np_qc[-2](alm2c, bem1c, wc)); aNmm3 = abs(Nm_qc[-3](alm2c, bem1c, wc))
    # Auxiliary constants from the proof of Lemma 5.7 (Steps 1 and 3):
    CQ = 2*(n-1)/(2*n-1)*(2*C_mu + aMu1**2 + 2*aMu1*C_mu/C_rad**2 + C_mu**2/C_rad**4) \
         + 2*(n-1)*(2*aEta0*C_eta + C_eta**2/C_rad**2)
    CDdes  = 4*(n-1)*C_al + 4*C_be**2/C_rad**2
    CNpdes = 2*(n-1)*C_al + 4*(n-1)*C_be**2/C_rad**2
    CNmdes = 4*C_al*C_be/C_rad**2 + 4*aB*C_al + 2*(n-1)*C_be/C_rad**2
    CQdes  = 2*(n-1)/(2*n-1)*(2*aMu1*C_mu + C_mu**2/C_rad**2) + 2*(n-1)*C_eta**2
    CD  = aD0.max(CDdes + 8*aB*C_be)
    CNp = aNp0.max(CNpdes + 8*(n-1)*aB*C_be)
    CNm = aNm1.max(CNmdes + 4*aA*C_be)
    # The C_{R_x} constants of Step 4:
    CRbe = aB*CQ/(4*C_rad**2) + aQ0*C_be/C_rad**2 + CQ*C_be/C_rad**4 + aEta0*C_al/2 \
           + C_al*C_eta/(2*C_rad**2) + 2*aB*C_mu/(2*C_rad**2) + 2*aMu1*C_be/(2*C_rad**2) \
           + 2*C_be*C_mu/(2*C_rad**4)
    CReta = (n-1)/C_rad**2*(C_eta*(aD2*aQ0 + aD2*CQ/C_rad**2 + CD) + aQ0*CD*C_eta/C_rad**2
                           + CD*CQ*C_eta/C_rad**4) \
           + (n-1)/4*CDdes*aEta0 + (n-1)*CD/4*C_eta/C_rad**2 \
           + ((2*n-1)*lam + aQ0)/4*CNmdes + aNmm3*CQdes/(4*C_rad**2) + CQ*CNm/(4*C_rad**2)
    CRmu = (n-1)/C_rad**2*(C_mu*(aD2*aQ0 + CD + aD2*CQ/C_rad**2) + aQ0*aMu1*CD + CD*CQ
                           + (aQ0*CD*C_mu + aMu1*CD*CQ)/C_rad**2 + CD*CQ*C_mu/C_rad**4) \
           + (n-1)/4*(2*aMu1 + aQ0)*CDdes + ((2*n-1)*lam + aQ0)/4*CNpdes \
           + ((n-1)*aD2*(aMu1*CQ + CQdes) + aNpm2*CQdes + CQ*CNp + (n-1)*CD*C_mu)/(4*C_rad**2)
    # Off-diagonal c-coefficients (m-independent) in absolute value:
    acetabe = abs(2*(n-1)*((n-1)*alm2c + bem1c**2)/alm2c)
    acmube = abs(2*(n-1)**2*(2*n-1)*bem1c/alm2c)
    acetamu = abs(16*(n-1)*alm2c*bem1c/(2*n-1))
    acmueta = abs(4*(n-1)**2*(2*n-1)*bem1c)
    # And now we check the four closure inequalities:
    closures = (
        ("(89)          ", aQ0 + CQ/C_rad**2 + 2*(CQ + C_mu)*aA/C_al + 2*aMu1
                           + 2*C_mu/C_rad**2
                           + (2*aB*C_eta + 2*aEta0*C_be + 2*C_be*C_eta/C_rad**2)/C_al,
                           C_rad**2),
        ("(94)           ", 8*((n-1)*aA + aB**2)*(CReta + acetabe*CRbe)/((n-1)*aD2**2)
                           + acetamu*(CRmu + acmube*CRbe)/(4*(n-1)**2*aD2**2),
                           C_eta),
        ("(95)           ", acmueta*(CReta + acetabe*CRbe)/(5*(n-1)**2*aD2**2)
                           + 9*((n-1)*aA + aB**2)*(CRmu + acmube*CRbe)/((n-1)*aD2**2),
                           C_mu),
        ("(96)           ", 2*aA*C_eta/5 + CRbe,
                           C_be),
    )
    for name, Lhs, Rhs in closures:
        if not Lhs.upper() <= Rhs.lower():
            if verbose: print("  FAIL closure %s:  LHS up to %s  vs  RHS from %s"
                              % (name, Lhs.upper(), Rhs.lower()))
            return False
        elif verbose:
            print("  %s holds with margin %.3f" % (name, float(Rhs.lower()/Lhs.upper())))
    return True

def derive_qc_constants(n, m, alm2c, bem1c, verbose=True):
    # Here, we search for constants C_al, C_be, C_eta, C_mu, C_rad satisfying the
    # hypothesis of Lemma 5.7 at the quadric endpoint.
    # This part is really irrelevant for the rigorous validity of the code, as long as we obtain
    # constants C_al, C_be, C_eta, C_mu, C_rad that return True under check_qc_constants, the proof is rigorous
    qc_coefficients(n, m)
    Dm2c = 4*((n-1)*alm2c - bem1c**2)
    wc = 1/Dm2c
    sups = {}
    for k in range(0, m-1, 2):
        sups[('al', k)] = abs(al_qc[k](alm2c, bem1c, wc))
        sups[('be', k)] = abs(be_qc[k+1](alm2c, bem1c, wc))
        sups[('eta', k)] = abs(eta_qc[k+2](alm2c, bem1c, wc))
        sups[('mu', k)] = abs(mu_qc[k+3](alm2c, bem1c, wc))
    # Special coefficients and off-diagonal c's on the balls (independent of C_rad)
    lam = 2*(n+1)
    aA = abs(alm2c); aB = abs(bem1c); aD2 = abs(Dm2c)
    aMu1 = abs(mu_qc[1](alm2c, bem1c, wc))
    aEta0 = abs(eta_qc[0](alm2c, bem1c, wc))
    aQ0 = abs(Q_qc[0](alm2c, bem1c, wc))
    acetabe = abs(2*(n-1)*((n-1)*alm2c + bem1c**2)/alm2c)
    acmube = abs(2*(n-1)**2*(2*n-1)*bem1c/alm2c)
    acetamu = abs(16*(n-1)*alm2c*bem1c/(2*n-1))
    acmueta = abs(4*(n-1)**2*(2*n-1)*bem1c)
    margin = RBF(11)/RBF(10)
    # We iterate dyadically over C_rad
    for e in range(1, 200):
        C_rad = RBF(1.5)**e
        Cs = {}
        # We set C_al, C_be, C_eta, C_mu from the bounds (87) with k <= m-2
        for x in ('al', 'be', 'eta', 'mu'):
            Cs[x] = margin*RBF(max((sups[(x, k)]/(RBF(catalan_number(k))*C_rad**k)).upper()
                                   for k in range(0, m-1, 2)))
        # We increase C_eta, C_mu, C_be if needed: the C_rad-free parts of the closures
        # (the C^infty quantities) must fit inside HALF of each constant, so that the
        # other half accommodates all the C_rad-suppressed terms (hence the factors 2)
        CRbe0 = aEta0*Cs['al']/2
        CReta0 = (n-1)/4*aEta0*(4*(n-1)*Cs['al']) + ((2*n-1)*lam + aQ0)/4*(4*aB*Cs['al'])
        CRmu0 = (n-1)/4*(2*aMu1 + aQ0)*(4*(n-1)*Cs['al']) \
                + ((2*n-1)*lam + aQ0)/4*(2*(n-1)*Cs['al'])
        LHSeta0 = 8*((n-1)*aA + aB**2)*(CReta0 + acetabe*CRbe0)/((n-1)*aD2**2) \
                 + acetamu*(CRmu0 + acmube*CRbe0)/(4*(n-1)**2*aD2**2)
        LHSmu0 = acmueta*(CReta0 + acetabe*CRbe0)/(5*(n-1)**2*aD2**2) \
                 + 9*((n-1)*aA + aB**2)*(CRmu0 + acmube*CRbe0)/((n-1)*aD2**2)
        if not Cs['eta'].lower() >= (2*LHSeta0).upper():
            Cs['eta'] = 2*RBF(LHSeta0.upper())
        if not Cs['mu'].lower() >= (2*LHSmu0).upper():
            Cs['mu'] = 2*RBF(LHSmu0.upper())
        LHSbe0 = 2*aA*Cs['eta']/5 + CRbe0
        if not Cs['be'].lower() >= (2*LHSbe0).upper():
            Cs['be'] = 2*RBF(LHSbe0.upper())
        # Now, we check if those constant choices work. If not, C_rad doubles
        if check_qc_constants(n, m, alm2c, bem1c, Cs['al'], Cs['be'],
                              Cs['eta'], Cs['mu'], C_rad, verbose=False):
            if verbose:
                print("SUCCESS at C_rad = 2^%d:  C_al=%.4g  C_be=%.4g  C_eta=%.4g  C_mu=%.4g"
                      % (e, float(Cs['al']), float(Cs['be']), float(Cs['eta']), float(Cs['mu'])))
                check_qc_constants(n, m, alm2c, bem1c, Cs['al'], Cs['be'],
                                   Cs['eta'], Cs['mu'], C_rad, verbose=True)  # print margins
            return Cs['al'], Cs['be'], Cs['eta'], Cs['mu'], C_rad
    if verbose: print("FAILED: no C_rad up to 2^30 works")
    return None

print("CELL 4: definitions ready. Run Example 4 below.", flush=True)

## Example 4 — verify the quadric bounds
Repeat the same argument at the other singular orbit, and check that $H_+=1200$
lies in the guaranteed convergence region. Each printed margin is the ratio
of a lower bound for the right-hand side to an upper bound for the left-hand
side of one closure inequality; a ratio at least 1 verifies that inequality.

In [ ]:
demo_qc_constants = derive_qc_constants(demo_n, demo_m_qc,
    demo_complex_box[2], demo_complex_box[3], verbose=False)
assert demo_qc_constants is not None, "No quadric constants certified."
for demo_label, demo_value in zip(("C_alpha", "C_beta", "C_eta", "C_mu", "C_rad"), demo_qc_constants):
    print(demo_label, "=", demo_value)
demo_qc_bounds_ok = check_qc_constants(demo_n, demo_m_qc,
    demo_complex_box[2], demo_complex_box[3], *demo_qc_constants, verbose=True)
assert demo_qc_bounds_ok
print("Quadric bounds verified on the complex parameter balls:", demo_qc_bounds_ok)
print("4 C_rad =", 4*demo_qc_constants[-1], "; chosen H_+ =", demo_Hp)
assert demo_Hp.lower() > (4*demo_qc_constants[-1]).upper()

In [ ]:
# CELL 5 
# Here we evaluate solutions at H with the series from RP^n. Given a ball containing b2 and u1 (called b2b and u1b)
# and a ball containing H (called Hb) as well as integers n, m, we compute the series until order m 
# use interval arithmetics to compute the series up to order m (elementary operations)
# and add an extra error term that rigorously accounts for truncation as in Propositions 5.5 and 5.8
# y indicates if we want to obtain a derivative (with respect to b2 or u1, or None) of the result
# and r has the same meaning as in Propositions 5.5 and 5.8
# Then, we do the same for the complex quadric 
# ========================================

def evaluate_rp_series(n, m, b2b, u1b, r, y, Hb, verbose=True):
    if y not in ('None', 'b2', 'u1'):
        if verbose: print("ERROR: y must be 'None', 'b2' or 'u1'")
        return None
        
    res = derive_rp_constants(n, m, CBF(b2b).add_error(r), CBF(u1b).add_error(r), verbose=False)
    
    if res is None:
        if verbose: print("ERROR: no certified constants on the complexified balls")
        return None
        
    C_a, C_b, C_u, C_v, C_rad = res
    if not Hb.upper() < (-4*C_rad).lower():
        if verbose: print("ERROR: H < -4 C_rad fails (C_rad = %.4g); not certified at H" % float(C_rad))
        return None

    # We load coefficients and the RBF ring
    rp_coefficients(n, m)
    Rng = b2.parent()
    
    # Tail check for Definition 6.1: D > 0 on (-inf, H_-]. By Proposition (5.5) with m = 3
    # |a - 1| <= |a_2| / H^2 + C_a (4 * C_rad)^4 / (|H_-|^3 (|H_-| - 4 C_rad) for all H < H_-
    # Similarly, |b| <= |b_2| / H^2 + C_b (4 * C_rad)^4 / (|H_-|^3 (|H_-| - 4 C_rad)
    # We use those ""rough"" enclosures to justify D \geq DI.lower() > 0 for all H < H_-
    aux = (RBF(4) * C_rad)**4 / (abs(Hb)**3 * (abs(Hb) - 4*C_rad))
    aI  = RBF(1).add_error((abs(Rng(a_rp[2])(b2b, u1b))*Hb**(-2) + C_a * aux).upper())
    bI  = RBF(0).add_error((abs(Rng(b_rp[2])(b2b, u1b))*Hb**(-2) + C_b * aux).upper())
    DI  = 2*(n-1)*(aI + bI) + 2*aI*bI - aI**2 - bI**2 - 1
    if not DI.lower() > 0:
        raise RuntimeError("D > 0 not certified on (-inf, H_-]")
        
    # We evaluate the truncated series on the balls and add the residual bound (71)
    # In contrast with the previous bit of code, we use ""accurate"" enclosures, by using the series up to order m
    out = {}
    for nm_, X, Cx in (('a', a_rp, C_a), ('b', b_rp, C_b), ('u', u_rp, C_u), ('v', v_rp, C_v)):
        if y == 'None':
            S = sum(Rng(X[kk])(b2b, u1b)*Hb**(-kk) for kk in X if kk <= m)
            out[nm_] = S.add_error((4**(m+1)*Cx*C_rad**(m+1) \
                                    /(abs(Hb)**m*(abs(Hb) - 4*C_rad))).upper())
        else:
            S = sum(Rng(X[kk]).derivative(b2 if y == 'b2' else u1)(b2b, u1b)*Hb**(-kk) \
                    for kk in X if kk <= m)
            out[nm_] = S.add_error((4**(m+1)*Cx*C_rad**(m+1) \
                                    /(r*abs(Hb)**m*(abs(Hb) - 4*C_rad))).upper())
    return out['a'], out['b'], out['u'], out['v']


# Now we do the same for the complex quadric endpoint
# The structure is analogous, we only comment differences relative to evaluate_rp_series
def evaluate_qc_series(n, m, alm2b, bem1b, r, y, Hb, verbose=True):
    if y not in ('None', 'alm2', 'bem1'):
        if verbose: print("ERROR: y must be 'None', 'alm2' or 'bem1'")
        return None

    res = derive_qc_constants(n, m, CBF(alm2b).add_error(r), CBF(bem1b).add_error(r), verbose=False)

    if res is None:
        if verbose: print("ERROR: no certified constants on the complexified balls")
        return None

    C_al, C_be, C_eta, C_mu, C_rad = res
    if not Hb.lower() > (4*C_rad).upper():
        if verbose: print("ERROR: H > 4 C_rad fails (C_rad = %.4g); not certified at H" % float(C_rad))
        return None

    # We load the coefficients. Recall w = 1/(4((n-1)*alm2 - bem1^2))
    qc_coefficients(n, m)
    Rng = alm2.parent()
    wb = 1/(4*((n-1)*alm2b - bem1b**2))

    # Tail check for Definition 6.1: D > 0 on [H_+, +inf), in the equivalent form
    # D/H^2 = 4(n-1)(al/H^2) - 4(be/H)^2 - 1/H^2. By Proposition 5.8 with m = 2,
    # |al/H^2 - alm2| <= |al_0|/H^2 + |al_2|/H^4 + C_al (4 C_rad)^3 / (|H|^4 (|H| - 4 C_rad)) and
    # |be/H - bem1|  <= |be_1|/H^2 + C_be (4 C_rad)^3 / (|H|^3 (|H| - 4 C_rad)) for all H >= H_+
    aux = (RBF(4) * C_rad)**3 / (abs(Hb)**2 * (abs(Hb) - 4*C_rad))
    alI = RBF(alm2b).add_error((abs(Rng(al_qc[0])(alm2b, bem1b, wb))*Hb**(-2) + abs(Rng(al_qc[2])(alm2b, bem1b, wb))*Hb**(-4) + C_al * aux * Hb**(-2)).upper())
    beI = RBF(bem1b).add_error((abs(Rng(be_qc[1])(alm2b, bem1b, wb))*Hb**(-2) + C_be * aux * Hb**(-1)).upper())
    DI  = 4*(n-1)*alI - 4*beI**2 - Hb**(-2)
    if not DI.lower() > 0:
        raise RuntimeError("D > 0 not certified on [H_+, +inf)")


    # We evaluate the truncated series on the balls and add the residual bound (97).
    # The coefficients are series in (alm2, bem1, w); since w stands for
    # 1/D_{-2} = 1/(4((n-1)*alm2 - bem1^2)), the derivatives with respect to alm2, bem1
    # carry the terms dw/dalm2 = -4(n-1)*w^2 and dw/dbem1 = 8*bem1*w^2
    out = {}
    for nm_, X, Cx in (('al', al_qc, C_al), ('be', be_qc, C_be), \
                       ('eta', eta_qc, C_eta), ('mu', mu_qc, C_mu)):
        if y == 'None':
            S = sum(Rng(X[kk])(alm2b, bem1b, wb)*Hb**(-kk) for kk in X if kk <= m)
            out[nm_] = S.add_error((4**(m+1)*Cx*C_rad**(m+1) \
                                    /(abs(Hb)**m*(abs(Hb) - 4*C_rad))).upper())
        else:
            S = sum((Rng(X[kk]).derivative(alm2 if y == 'alm2' else bem1)(alm2b, bem1b, wb) \
                     + Rng(X[kk]).derivative(w)(alm2b, bem1b, wb) \
                       *(-4*(n-1)*wb**2 if y == 'alm2' else 8*bem1b*wb**2))*Hb**(-kk) \
                    for kk in X if kk <= m)
            out[nm_] = S.add_error((4**(m+1)*Cx*C_rad**(m+1) \
                                    /(r*abs(Hb)**m*(abs(Hb) - 4*C_rad))).upper())
    return out['al'], out['be'], out['eta'], out['mu']


print("CELL 5: definitions ready. Run Example 5 below.", flush=True)

## Example 5 — evaluate the series, including their tails
At the candidate centre, compute rigorous starting data at $H=-800$ and
$H=1200$. Each ball includes the truncation error. Then display the four
components of $\partial_{b_2}(a,b,u,v)$ at $H=-800$, using the Cauchy tail bound.
Note the function's interface uses the **string** `'None'` for a value evaluation.

In [ ]:
demo_rp_start = evaluate_rp_series(demo_n, demo_m_rp,
    demo_p0[0], demo_p0[1], RBF(0), 'None', demo_Hm, verbose=False)
demo_qc_start = evaluate_qc_series(demo_n, demo_m_qc,
    demo_p0[2], demo_p0[3], RBF(0), 'None', demo_Hp, verbose=False)
assert demo_rp_start is not None and demo_qc_start is not None
print("RP starting data at H = -800, including the tail:")
for demo_label, demo_value in zip(("a", "b", "u", "v"), demo_rp_start):
    print(demo_label, "=", demo_value)
print("\nQuadric starting data at H = 1200, including the tail:")
for demo_label, demo_value in zip(("alpha", "beta", "eta", "mu"), demo_qc_start):
    print(demo_label, "=", demo_value)
demo_rp_b2 = evaluate_rp_series(demo_n, demo_m_rp,
    demo_p0[0], demo_p0[1], demo_r, 'b2', demo_Hm, verbose=False)
assert demo_rp_b2 is not None
print("\nParameter derivatives at H = -800, including the Cauchy tail:")
for demo_label, demo_value in zip(("d a/d b2", "d b/d b2", "d u/d b2", "d v/d b2"), demo_rp_b2):
    print(demo_label, "=", demo_value)

In [ ]:
#=======================================================
# CELL 6
# This cell contains the ODE running from H = H_- to H = 0 or from H = H_+ to H = 0
# In order to do that, we use a the CAPD library, which requires C++, so we use a driver
#=======================================================
import subprocess, os, shutil

# ---- EDIT THESE THREE IF YOUR SETUP DIFFERS -------------------------
CXX         = os.environ.get("RP_CXX", "/usr/bin/g++")
CAPD_PREFIX = os.path.expanduser(os.environ.get("CAPD_PREFIX", "~/capd"))
GMP_PREFIX  = ""  # Linux hosting; GMP and MPFR must already be installed.
# ---------------------------------------------------------------------

EXTRA_CF = ["-I"+GMP_PREFIX+"/include"] if GMP_PREFIX else []
EXTRA_LF = ["-L"+GMP_PREFIX+"/lib"]     if GMP_PREFIX else []
import tempfile
_capd_demo_dir = tempfile.TemporaryDirectory(prefix="cpn_capd_")
DRIVER_SRC = os.path.join(_capd_demo_dir.name, "cpn_capd_leg.cpp")
DRIVER_BIN = os.path.join(_capd_demo_dir.name, "cpn_capd_leg")

# This is the ODE in (38)
def capd_rp_field():
    H  = "(Hs+sg*s)"
    Q  = "(2*(nn+1)+(({H}*{H})+nn*(nn-1)*(u*u+v*v)-2*(nn-1)*(nn-1)*u*v)/(2*nn-1))".format(H=H)
    D  = "(2*(nn-1)*(a+b)+2*a*b-a*a-b*b-1)"
    N1 = "(nn*a*a+(nn-2)*b*b-2*(nn-1)*a*b+2*(nn-1)*b-nn)"
    N2 = "((nn-2)*a*a+nn*b*b-2*(nn-1)*a*b+2*(nn-1)*a-nn)"
    sc = "(({H}*{H})-{Q}+2*(2*nn-1)*(nn+1))".format(H=H, Q=Q)
    f  = ["sg*(2*a*u/{Q})", "sg*(2*b*v/{Q})",
          "sg*(((nn-1)*{D}*{H}*u+{sc}*{N1})/((nn-1)*{D}*{Q}))",
          "sg*(((nn-1)*{D}*{H}*v+{sc}*{N2})/((nn-1)*{D}*{Q}))"]
    return "time:s;par:nn,Hs,sg;var:a,b,u,v;fun:" + \
           ",".join(p.format(Q=Q, D=D, N1=N1, N2=N2, sc=sc, H=H) for p in f) + ";"
    
# This is the ODE in complex quadric variables from Remark 4.5
def capd_qc_field():
    H  = "(Hs+sg*s)"
    Q  = "(2*(nn+1)+(({H}*{H})+2*(nn-1)*mu*mu+2*(nn-1)*(2*nn-1)*eta*eta)/(2*nn-1))".format(H=H)
    D  = "(4*(nn-1)*al-4*be*be-1)"
    Np = "(2*(nn-1)*al+4*(nn-1)*be*be-nn)"
    Nm = "(4*al*be-2*(nn-1)*be)"
    # scal = H^2 - Q + (2n-1)*lam, grouped so the mu ~ H cancellation is algebraic:
    sc = "(2*(nn-1)*(({H}-mu)*({H}+mu))/(2*nn-1)-2*(nn-1)*eta*eta+4*(nn-1)*(nn+1))".format(H=H)
    f  = ["sg*(2*(al*mu+be*eta)/{Q})", "sg*(2*(al*eta+be*mu)/{Q})",
          "sg*(((nn-1)*{D}*{H}*eta+{sc}*{Nm})/((nn-1)*{D}*{Q}))",
          "sg*(((nn-1)*{D}*{H}*mu+{sc}*{Np})/((nn-1)*{D}*{Q}))"]
    return "time:s;par:nn,Hs,sg;var:al,be,eta,mu;fun:" + \
           ",".join(p.format(Q=Q, D=D, Np=Np, Nm=Nm, sc=sc, H=H) for p in f) + ";"

CPP = r'''#include "capd/mpcapdlib.h"
#include <cstdio>
#include <cstdlib>
#include <iostream>
#include <sstream>
#include <stdexcept>
using namespace capd;
static const char* FIELD  = "@FIELD@";
static const char* FIELDQ = "@FIELDQ@";
// argv: FID n Hstart Hend ORDER TOL BITS  then 8 decimal interval endpoints
// printed endpoints are widened outward by relative 1e-38
static MpFloat widen_lo(const MpFloat& x){ MpFloat w=abs(x)*MpFloat(1e-38)+MpFloat(1e-200); return x-w; }
static MpFloat widen_hi(const MpFloat& x){ MpFloat w=abs(x)*MpFloat(1e-38)+MpFloat(1e-200); return x+w; }
static void pr(const char* tag,int i,int j,const MpInterval& v){
  std::ostringstream o; o.precision(45);
  o<<tag<<" "<<i; if(j>=0) o<<" "<<j;
  o<<" "<<widen_lo(v.leftBound())<<" "<<widen_hi(v.rightBound());
  std::cout<<o.str()<<"\n";
}
static MpInterval Dval(int fid, double nn, const MpIVector& z){
  MpInterval N(nn), one(1.0), two(2.0), four(4.0);
  return fid ? ( four*(N-one)*z[0] - four*z[1]*z[1] - one )
             : ( two*(N-one)*(z[0]+z[1]) + two*z[0]*z[1] - z[0]*z[0] - z[1]*z[1] - one );
}
int main(int argc, char* argv[]) {
  try {
    if (argc < 16) { fprintf(stderr, "ERR: usage\n"); return 2; }
    int fid=atoi(argv[1]);
    double nn=strtod(argv[2],0), Hs=strtod(argv[3],0), He=strtod(argv[4],0);
    int order=atoi(argv[5]); double tol=strtod(argv[6],0);
    MpFloat::setDefaultPrecision(atoi(argv[7]));
    MpIVector x(4);
    for (int i=0;i<4;i++){
      MpFloat lo(argv[8+2*i]), hi(argv[9+2*i]);
      x[i]=MpInterval(widen_lo(lo),widen_hi(hi));
    }
    double sg=(He>=Hs)?1.0:-1.0; MpFloat L=abs(MpFloat(He)-MpFloat(Hs));
    MpIMap vf(fid?FIELDQ:FIELD);
    vf.setParameter("nn",MpInterval(nn)); vf.setParameter("Hs",MpInterval(Hs)); vf.setParameter("sg",MpInterval(sg));
    MpIOdeSolver solver(vf,order);
    solver.setAbsoluteTolerance(tol); solver.setRelativeTolerance(tol);
    MpITimeMap timeMap(solver);
    MpIVector r(4); MpIMatrix M(4,4);
    MpC1Rect2Set s(x);
    long nst=0;
    int welldef=1; MpFloat Dmin(1e300);
    { MpFloat d0=Dval(fid,nn,x).leftBound();
      if (!(d0 > MpFloat(0.0))) welldef=0; else if (d0 < Dmin) Dmin=d0; }
    timeMap.stopAfterStep(true);
    do {
      r=timeMap(MpInterval(L),s,M); nst++;
      { const MpIOdeSolver::SolutionCurve& cur = solver.getCurve();
        MpIVector enc = cur(MpInterval(cur.getLeftDomain(), cur.getRightDomain()));
        MpFloat d=Dval(fid,nn,enc).leftBound();
        if (!(d > MpFloat(0.0))) welldef=0; else if (d < Dmin) Dmin=d; }
      if (nst%40==0 || timeMap.completed()) {
        MpFloat w(0); for (int i=0;i<4;i++){ MpFloat d=r[i].rightBound()-r[i].leftBound(); if(d>w)w=d; }
        std::ostringstream o; o.precision(10);
        o<<"H="<<(MpFloat(Hs)+MpFloat(sg)*s.getCurrentTime().rightBound());
        o.precision(3); o<<"  diam="<<w<<"  steps="<<nst<<"\n";
        std::cerr<<o.str(); std::cerr.flush();
      }
    } while (!timeMap.completed());
    for (int i=0;i<4;i++) pr("STATE",i,-1,r[i]);
    for (int i=0;i<4;i++) for (int j=0;j<4;j++) pr("MAT",i,j,M[i][j]);
    { std::ostringstream o; o.precision(20);
      o<<"WELLDEF "<<welldef<<"\n"<<"EPS "<<Dmin<<"\n"; std::cout<<o.str(); }
    printf("OK\n"); return 0;
  } catch (std::exception& e) { fprintf(stderr,"ERR: %s\n",e.what()); return 1; }
    catch (...)               { fprintf(stderr,"ERR: unknown\n"); return 1; }
}
'''
def build_capd_driver(force=False):
    # the stamp records the source of the LAST SUCCESSFUL compile: a failed or
    # stale build can never be mistaken for a current binary
    src=CPP.replace("@FIELD@",capd_rp_field()).replace("@FIELDQ@",capd_qc_field())
    stamp=DRIVER_BIN+".src"
    if not force and os.path.exists(DRIVER_BIN) and os.path.exists(stamp) \
       and open(stamp).read()==src: return
    _LEG_CACHE.clear()
    open(DRIVER_SRC,"w").write(src)
    pref=CAPD_PREFIX
    tries=[]
    for c in (os.environ.get("CAPD_CONFIG"),
              os.path.join(pref,"bin","capd-config"), shutil.which("capd-config")):
        if c and os.path.exists(c):
            cf=subprocess.run([c,"--cflags"],capture_output=True,text=True).stdout.split()
            lf=subprocess.run([c,"--libs"],capture_output=True,text=True).stdout.split()
            if lf: tries.append((cf,lf))
    pc=os.path.join(pref,"lib","pkgconfig","capd.pc")
    if os.path.exists(pc):                       # mini pkg-config parse
        vars={}
        for line in open(pc):
            line=line.strip()
            if "=" in line and ":" not in line.split("=",1)[0]:
                k,v=line.split("=",1); vars[k.strip()]=v.strip()
        def _sub(t):
            while "${" in t:
                i=t.index("${"); j=t.index("}",i); t=t[:i]+vars.get(t[i+2:j],"")+t[j+1:]
            return t
        cf=[]; lf=[]
        for line in open(pc):
            if line.startswith("Cflags:"): cf=_sub(line.split(":",1)[1]).split()
            if line.startswith("Libs:"):   lf=_sub(line.split(":",1)[1]).split()
        if lf: tries.append((cf,lf))
    tries.append((["-I"+os.path.join(pref,"include")],
                  ["-L"+os.path.join(pref,"lib"),"-lcapd"]))
    errs=[]
    for cf,lf in tries:                          # try-compile until one works
        r=subprocess.run([CXX,"-O2","-std=c++17",DRIVER_SRC]+cf+EXTRA_CF
                         +["-o",DRIVER_BIN]+lf+EXTRA_LF+["-lmpfr","-lgmp"],
                         capture_output=True,text=True)
        if r.returncode==0:
            open(stamp,"w").write(src); return
        lines=(r.stderr.strip() or "?").splitlines()
        errs.append(next((l for l in lines if "error" in l.lower()), lines[0]))
    raise RuntimeError("compile failed; attempts:\n"+"\n".join(errs)
                       +"\n(set CAPD_CONFIG or CAPD_PREFIX to your MPFR-enabled CAPD build)")
_RIF256=RealIntervalField(256)
_RF200=RealField(220)
def _decpair(x):
    I=_RIF256(x); return (repr(I.lower()), repr(I.upper()))
def _fromdec(lo,hi):
    I=_RIF256(_RF200(lo),_RF200(hi))
    w=abs(I).upper()*_RF200(1e-50)+_RF200(1e-280)
    return I+_RIF256(-w,w)
def _run(n,fid,x,H0,H1,order,tol,bits,verbose):
    build_capd_driver()
    args=[DRIVER_BIN,str(fid),repr(float(n)),repr(float(H0)),repr(float(H1)),
          str(order),repr(float(tol)),str(bits)]
    for iv in x: args+=list(_decpair(iv))
    p=subprocess.Popen(args,stdout=subprocess.PIPE,stderr=subprocess.PIPE,text=True)
    for line in p.stderr:
        if verbose: print(line,end="",flush=True)   # live H/diam progress
    out=p.stdout.read(); p.wait()
    if p.returncode!=0 or "OK" not in out:
        raise RuntimeError("driver failed (see lines above):\n"+out)
    st=[None]*4; M=[[None]*4 for _ in range(4)]
    wd=None; eps=None                             
    for line in out.strip().splitlines():
        t=line.split()
        if   t[0]=="STATE": st[int(t[1])]=RBF(_fromdec(t[2],t[3]))
        elif t[0]=="MAT":   M[int(t[1])][int(t[2])]=RBF(_fromdec(t[3],t[4]))
        elif t[0]=="WELLDEF": wd=(t[1]=="1")       
        elif t[0]=="EPS":     eps=t[1]             
    if wd is None:
        raise RuntimeError("driver did not report WELLDEF (stale binary?); "
                           "rerun build_capd_driver(force=True)")
    if verbose: print("   D > 0 verified: %s   (epsilon = %s)" % (wd, eps))
    return st,M,wd                                
_LEG_CACHE = {}
def run_ode_leg(mode, x, H0, H1, base=None, n=None, order=24, tol=1e-14, bits=128, verbose=True):
    """Rigorous multiprecision leg H0 -> H1.
    """
    if mode not in ('rp_nd', 'rp_b2', 'rp_u1', 'qc_nd', 'qc_alm2', 'qc_bem1'):
        raise ValueError("unknown mode %r" % (mode,))
    if n is None: n = globals()['n']
    if bits < 128: raise ValueError("bits >= 128 (the 1e-38 I/O widening assumes it)")
    deriv = not mode.endswith('_nd')
    if deriv and base is None:
        raise ValueError("derivative modes need base = state 4-vector at H0")
    for Hx in (H0, H1):
        B = RBF(Hx)
        if not (B.rad() == 0 and float(B.lower()) == float(B.upper())):
            raise ValueError("H0/H1 must be exact float64 grid values (e.g. RBF(-1000.0))")
    H0f, H1f = float(RBF(H0).mid()), float(RBF(H1).mid())
    fid = 0 if mode[:2] == 'rp' else 1
    y = [RBF(v) for v in (base if deriv else x)]
    key = (fid, float(n), tuple(_decpair(v) for v in y), H0f.hex(), H1f.hex(),
           int(order), float(tol), int(bits))
    if key not in _LEG_CACHE:
        _LEG_CACHE[key] = _run(n, fid, y, H0f, H1f, order, tol, bits, verbose)
    st, M, wd = _LEG_CACHE[key]                   
    if not deriv:
        return st, wd                              
    xB = [RBF(v) for v in x]
    return [sum(M[i][j]*xB[j] for j in range(4)) for i in range(4)], wd 

# We compile the driver now, so that a wrong setting above is reported by this
# cell instead of surfacing much later, in the middle of the certification.
build_capd_driver(force=True)
print("CAPD driver compiled OK ->", DRIVER_BIN)


print("CELL 6: definitions ready. Run Example 6 below.", flush=True)

## Example 6 — propagate one solution to H = 0 (slower)
The preceding definition cell compiles the CAPD driver. Now use it to integrate
from the RP starting data to the matching orbit. The returned flag records the
positivity check needed along the integration. This calculation is cached in
the current kernel and can be reused by Example 7.

In [ ]:
import time
demo_step_started = time.time()
print("Integrating the RP solution from H = -800 to H = 0 ...", flush=True)
demo_rp_zero, demo_rp_path_ok = run_ode_leg('rp_nd', demo_rp_start,
    demo_Hm, RBF(0), n=demo_n, order=21, tol=1e-17, bits=128, verbose=False)
for demo_label, demo_value in zip(("a(0)", "b(0)", "u(0)", "v(0)"), demo_rp_zero):
    print(demo_label, "=", demo_value)
print("Positivity check along the RP integration:", demo_rp_path_ok)
assert demo_rp_path_ok, "Required positivity was not verified."
print("Elapsed seconds:", round(time.time() - demo_step_started, 1))

In [ ]:
# CELL 7.
# Here we define our matching function 'matching_function_nd' as a substraction of 
# rp_solve_nd (which takes b2 and u1, as Real Ball Fields, and outputs the H = 0 value of the ODE 
# and qc_solve_nd (which takes alpha_{-2} and beta_{-1} as inputs and outputs the H = 0 values of the ODE
# We do the same for the Jacobian of the matching function, in that case using the ODE that arises from taking
# formal derivatives with respect to b2, u1, alpha_{-2} or beta_{-1} respectively.
# ===================

def rp_solve_nd(n, m_rp, b2b, u1b, H, order = 24, tol = 1e-14, bits = 128, verbose = False):
    xH    = evaluate_rp_series(n, m_rp, b2b, u1b, RBF(0.0), 'None', H, verbose = verbose)
    if verbose: print("Series computation finished")
    # We take data x(H) = xH and solve from H until 0 
    x0, well_def    = run_ode_leg('rp_nd', xH, H, RBF(0.0), n=n, order=order, tol=tol, bits=bits, verbose=verbose)
    return x0, well_def

def qc_solve_nd(n, m_qc, alm2b, bem1b, H, order = 24, tol = 1e-14, bits = 128, verbose = False):
    xH      = evaluate_qc_series(n, m_qc, alm2b, bem1b, RBF(0.0), 'None', H, verbose = verbose)
    if verbose: print("Series computation finished")
    # We take data x(H) = xH and solve from H until 0 
    x0, well_def      = run_ode_leg('qc_nd', xH, H, RBF(0.0), n=n, order=order, tol=tol, bits=bits, verbose=verbose)
    return x0, well_def

def matching_function_nd(n, m_rp, m_qc, b2b, u1b, alm2b, bem1b, Hm, Hp, order_rp = 24, order_qc = 24, tol_rp = 1e-30, tol_qc = 1e-30, bits = 128, verbose = False):
    rp_0, well_def1 = rp_solve_nd(n, m_rp, b2b, u1b, Hm, order = order_rp, tol = tol_rp, bits = bits, verbose = verbose)
    qc_0, well_def2 = qc_solve_nd(n, m_qc, alm2b, bem1b, Hp, order = order_qc, tol = tol_qc, bits = bits, verbose = verbose)
    aux = [qc_0[0] + qc_0[1], qc_0[0] - qc_0[1], qc_0[3] + qc_0[2], qc_0[3] - qc_0[2]]
    res = [rp_0[i] - aux[i] for i in range(4)]
    if well_def1 and well_def2:
        print("The matching function G is well-defined on the whole box X")
        return res
    print("Error, matching function not well-defined")
    return None

    
def rp_solve_jac(n, m_rp, b2b, u1b, H, r, order = 24, tol = 1e-14, bits = 128, verbose = False):
    xH    = evaluate_rp_series(n, m_rp, b2b, u1b, r, 'None', H, verbose = verbose)
    xH_b2 = evaluate_rp_series(n, m_rp, b2b, u1b, r, 'b2',   H, verbose = verbose)
    xH_u1 = evaluate_rp_series(n, m_rp, b2b, u1b, r, 'u1',   H, verbose = verbose)
    if verbose: print("Series computation finished succesfully")
    x0_b2, wd_b2 = run_ode_leg('rp_b2', xH_b2, H, RBF(0.0), base=xH,  n=n, order=order, tol=tol, bits=bits, verbose=verbose) 
    x0_u1, wd_u1 = run_ode_leg('rp_u1', xH_u1, H, RBF(0.0), base=xH,  n=n, order=order, tol=tol, bits=bits, verbose=verbose)
    return x0_b2, x0_u1, (wd_b2 and wd_u1)                                

def qc_solve_jac(n, m_qc, alm2b, bem1b, H, r, order = 24, tol = 1e-14, bits = 128, verbose = False):
    xH      = evaluate_qc_series(n, m_qc, alm2b, bem1b, r, 'None', H)
    xH_alm2 = evaluate_qc_series(n, m_qc, alm2b, bem1b, r, 'alm2', H)
    xH_bem1 = evaluate_qc_series(n, m_qc, alm2b, bem1b, r, 'bem1', H)
    if verbose: print("Series computation finished")
    x0_alm2, wd_al = run_ode_leg('qc_alm2', xH_alm2, H, RBF(0.0), base=xH,  n=n, order=order, tol=tol, bits=bits, verbose=verbose)
    x0_bem1, wd_be = run_ode_leg('qc_bem1', xH_bem1, H, RBF(0.0), base=xH,  n=n, order=order, tol=tol, bits=bits, verbose=verbose)
    return x0_alm2, x0_bem1, (wd_al and wd_be)                            

def matching_function_jac(n, m_rp, m_qc, b2b, u1b, alm2b, bem1b, Hm, Hp, r = RBF(1e-3), order_rp = 24, order_qc = 24, tol_rp = 1e-14, tol_qc = 1e-14, bits = 128, verbose = False):
    rp_b2, rp_u1, well_def1 = rp_solve_jac(n, m_rp, b2b, u1b, Hm, r, tol = tol_rp, order = order_rp, bits = bits, verbose = verbose)
    qc_alm2, qc_bem1, well_def2 = qc_solve_jac(n, m_qc, alm2b, bem1b, Hp, r, tol = tol_qc, order = order_qc, bits = bits, verbose = verbose)
    T = lambda x: [x[0] + x[1], x[0] - x[1], x[3] + x[2], x[3] - x[2]]
    jac = [rp_b2, rp_u1, [-w for w in T(qc_alm2)], [-w for w in T(qc_bem1)]]
    if well_def1 and well_def2:                                           
        return jac
    print("Error, matching function not well-defined on the box X")
    return None


print("CELL 7: definitions ready. Run Example 7 below.", flush=True)

## Example 7 — display the matching error (slower)
Propagate both sides and subtract their values at $H=0$ in the common
$(a,b,u,v)$ coordinates. This gives an enclosure of $\mathcal G(p_0)$ at the
candidate centre. A small mismatch alone is **not** a proof that a zero exists;
the final example also controls the Jacobian throughout the box.

In [ ]:
demo_step_started = time.time()
print("Computing the four components of G at the candidate centre ...", flush=True)
demo_matching = matching_function_nd(demo_n, demo_m_rp, demo_m_qc,
    *demo_p0, demo_Hm, demo_Hp, order_rp=21, order_qc=26,
    tol_rp=1e-17, tol_qc=1e-22, bits=128, verbose=False)
assert demo_matching is not None, "The matching calculation failed."
for demo_label, demo_value in zip(("G_a", "G_b", "G_u", "G_v"), demo_matching):
    print(demo_label, "=", demo_value)
print("These are bounds at the candidate centre; certification comes next.")
print("Elapsed seconds:", round(time.time() - demo_step_started, 1))

In [ ]:
# CELL 8
# We first store our numerical values for b2, u1, alm2 and bem1 depending on the dimension
# They correspond to Table 1 in the paper
b2_cand   = {3: RBF("21.090615289608624"), 4: RBF("39.073179213574893"),
             5: RBF("69.506625525589775"), 6: RBF("120.06632755687508"),
             7: RBF("229.69165123519819")}
u1_cand   = {3: RBF("4.2769888845669771"), 4: RBF("6.0328404911983641"),
             5: RBF("8.4713521979792653"), 6: RBF("12.289082277882205"),
             7: RBF("20.840270491124821")}
alm2_cand = {3: RBF("0.0248433475704649"), 4: RBF("0.0559943268417511"),
             5: RBF("0.0750978465264258"), 6: RBF("0.0868218947390250"),
             7: RBF("0.0944735712466017")}
bem1_cand = {3: RBF("-0.045831878718430"), 4: RBF("-0.042848398677701"),
             5: RBF("-0.031246038856108"), 6: RBF("-0.020103748103872"),
             7: RBF("-0.010001596495958")}

# krawczyk_check tries to apply Krawczyk's Theorem (Theorem 6.4 in the paper) with the specified hyperparameters (tolerance and order of ODEsolvers,
# Hm and Hp correspond to H_- and H_+ in the paper, and r is the radius for the derivative estimate of Propositions 5.5 and 5.8 m corresponds to the order of our series
def krawczyk_check(nval, m_rp, m_qc, delta, Hm, Hp, r = RBF("1e-4"), tol_rp0 = 1e-30, tol_rpd = 1e-20, tol_qc0 = 1e-30, tol_qcd = 1e-20, order_rp0 = 24, order_rpd = 24, order_qc0 = 24, order_qcd = 24, verbose = True):

    if nval < 3 or nval > 7:
        print("ERROR: incorrect n")
        return None

    x0 = [b2_cand[nval], u1_cand[nval], alm2_cand[nval], bem1_cand[nval]]

    # We compute the result of the matching function on x and enclose the Jacobian on a delta-sized cube around x
    if verbose: print("Starting the check for n = ", nval)
    res = matching_function_nd(nval, m_rp, m_qc, RBF(x0[0]), RBF(x0[1]), RBF(x0[2]), RBF(x0[3]), Hm, Hp, tol_rp = tol_rp0, tol_qc = tol_qc0, order_rp = order_rp0, order_qc = order_qc0, verbose = verbose)
    if verbose: print("We enclosed G(x) by ", res)
    jac = matching_function_jac(nval, m_rp, m_qc, RBF(x0[0]).add_error(delta), RBF(x0[1]).add_error(delta),
                                   RBF(x0[2]).add_error(delta), RBF(x0[3]).add_error(delta), Hm, Hp, r = r, tol_rp = tol_rpd, tol_qc = tol_qcd, order_rp = order_rpd, order_qc = order_qcd, verbose = verbose)
    if verbose: print("We enclosed DG(x') for x' in X by ", jac)

    # We pick our A to be an approximate inverse of jac (see Remark )
    A   = matrix(RBF, matrix(RDF, [[RDF(jac[j][i].mid()) for j in range(4)]
                                   for i in range(4)]).inverse())
    if verbose: print("Our A is ", A)

    # We assemble DG from jac
    DG  = matrix(RBF, [[jac[j][i] for j in range(4)] for i in range(4)])

    # We bound ||Id - A * DG(x)|| and ||A * G(x0)||
    E   = identity_matrix(RBF, 4) - A*DG
    Ag  = A*vector(RBF, res)
    nrm = max(sum(abs(E[i, j]).upper() for j in range(4)) for i in range(4))
    g   = max(abs(Ag[i]).upper() for i in range(4))

    # We check condition (100)
    ok  = (nrm < 1) and (g + nrm * delta.upper() < delta.lower())   # nrm < 1 also certifies that A is invertible, as Theorem 6.4 requires
    print("      |A.G(mid)| <= %.3e   ||I - A.DG(X)||_inf <= %.3f   delta = %.0e   -->  %s"
          % (g, nrm, delta, "CERTIFIED: unique zero in the box" if ok else "NOT certified"))
    if nrm > 1:
        print("Need to refine jac")
    if g > delta.upper():
        print("Need to refine G(x0)")

    return ok

print("CELL 8: definitions ready. Run Example 8 below.", flush=True)

## Example 8 — certify the zero (full calculation; slower)
Run the paper's complete **n = 4** check. It bounds $\|A\mathcal G(p_0)\|_\infty$
and $\|I-A D\mathcal G(X)\|_{\infty\to\infty}$, then tests the Krawczyk
inequality. The original certification call below is unchanged and **clears
the ODE cache**, so this is a fresh calculation even after Examples 6–7.
Only a successful result from this cell is the certification.

In [ ]:
print("SageMath:", version(), flush=True)
import time

_LEG_CACHE.clear()
started = time.time()
print("Starting a fresh certification for n = 4", flush=True)
certified = krawczyk_check(4, 35, 16, RBF("1e-11"), RBF(-800), RBF(1200), order_rp0 = 21, tol_rp0 = 1e-17, tol_qc0 = 1e-22, order_qc0 = 26, tol_rpd = 1e-9, order_rpd = 11, tol_qcd = 1e-18, order_qcd = 22, verbose = False)
print("Elapsed seconds:", round(time.time() - started, 1), flush=True)
if certified:
    print("RESULT: CERTIFIED — a unique zero exists in the specified box.", flush=True)
else:
    print("RESULT: NOT CERTIFIED — this run did not establish the conclusion.", flush=True)